In [1]:
%pip install pymysql
%pip install sqlalchemy

  Using cached pymysql-1.2.0-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.2.0-py3-none-any.whl (45 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="XXXXXXXXXX",
    host="localhost",
    port=3306,
    database="new_schema"
)

engine = create_engine(connection_url)

df = pd.read_sql(
    "SELECT * FROM superstore_staging_0",
    engine
)

print(df.head())
print(df.shape)

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [3]:
df = df.drop(columns=['Row ID','Order ID','Customer ID','Customer Name','City','Product Name','Product ID','Country'])


In [4]:

high_threshold = df.loc[df["Profit"] > 0, "Profit"].median()

print(high_threshold)

13.452


In [5]:
def profit_class(profit):
    if profit < 0:
        return 0
    elif profit < high_threshold:
        return 1
    else:
        return 2

df["Profit Class"] = df["Profit"].apply(profit_class)

In [6]:
df = df.drop(columns=['Postal Code'])

In [7]:
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

# Month when customer ordered
df["Order Month"] = df["Order Date"].dt.month

# Number of days between order and shipping
df["Delivery Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Remove original dates
df.drop(columns=["Order Date", "Ship Date"], inplace=True)

In [8]:

label = df['Profit Class']
df

,Ship Mode,Segment,State,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit,Profit Class,Order Month,Delivery Days
0,Second Class,Consumer,Kentucky,South,Furniture,Bookcases,261.9600,2,0.00,41.9136,2,11,3
1,Second Class,Consumer,Kentucky,South,Furniture,Chairs,731.9400,3,0.00,219.5820,2,11,3
2,Second Class,Corporate,California,West,Office Supplies,Labels,14.6200,2,0.00,6.8714,1,6,4
3,Standard Class,Consumer,Florida,South,Furniture,Tables,957.5775,5,0.45,-383.0310,0,10,7
4,Standard Class,Consumer,Florida,South,Office Supplies,Storage,22.3680,2,0.20,2.5164,1,10,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19383,Second Class,Consumer,Florida,South,Furniture,Furnishings,25.2480,3,0.20,4.1028,1,1,2
19384,Standard Class,Consumer,California,West,Furniture,Furnishings,91.9600,2,0.00,15.6332,2,2,5
19385,Standard Class,Consumer,California,West,Technology,Phones,258.5760,2,0.20,19.3932,2,2,5
19386,Standard Class,Consumer,California,West,Office Supplies,Paper,29.6000,4,0.00,13.3200,1,2,5


In [9]:
df.drop(columns='Profit Class', inplace=True)


In [10]:
categorical_cols = [
    "Ship Mode",
    "Segment",
    "State",
    "Region",
    "Category",
    "Sub-Category",
    "Order Month"
]

df_0 = pd.get_dummies(
    df,
    columns=categorical_cols,
    dtype=int
)

In [11]:
import torch 

In [12]:
X_tensor = torch.tensor(df_0.values, dtype=torch.float32)
y_tensor = torch.tensor(label.values, dtype=torch.long)


In [13]:
X_tensor.size()

torch.Size([19388, 97])

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor,
    y_tensor,
    test_size=0.2,
    random_state=42,
    stratify=y_tensor
)

In [17]:
X_train.size()

torch.Size([15510, 97])

In [22]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# -------------------------
# DataLoaders
# -------------------------
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# -------------------------
# Model
# -------------------------
model = nn.Sequential(
    nn.Linear(97, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.25),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),
    nn.Dropout(0.20),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 3)
)

# -------------------------
# Loss and optimizer
# -------------------------
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

# Reduce LR if validation loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)

# -------------------------
# Early stopping settings
# -------------------------
epochs = 500

patience = 20
patience_counter = 0

best_val_loss = float("inf")

# -------------------------
# Training
# -------------------------
for epoch in range(epochs):

    # ===== TRAIN =====
    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        outputs = model(X_batch)

        loss = loss_fn(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===== VALIDATION =====
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            outputs = model(X_batch)

            loss = loss_fn(outputs, y_batch)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)

    val_loss /= len(val_loader)

    val_accuracy = correct / total

    # Update learning rate
    scheduler.step(val_loss)

    # -------------------------
    # Early stopping
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        patience_counter = 0

        # Save best model
        torch.save(model.state_dict(), "best_model.pth")

    else:
        patience_counter += 1

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy * 100:.2f}% | "
            f"Patience: {patience_counter}/{patience}"
        )

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break


# -------------------------
# Load best model
# -------------------------
model.load_state_dict(torch.load("best_model.pth"))

print("\nBest model loaded.")

Epoch   5 | Train Loss: 0.2657 | Val Loss: 0.3382 | Val Acc: 83.52% | Patience: 1/20
Epoch  10 | Train Loss: 0.2436 | Val Loss: 0.3313 | Val Acc: 85.02% | Patience: 6/20
Epoch  15 | Train Loss: 0.2073 | Val Loss: 0.6238 | Val Acc: 65.27% | Patience: 2/20
Epoch  20 | Train Loss: 0.2122 | Val Loss: 0.1604 | Val Acc: 96.18% | Patience: 0/20
Epoch  25 | Train Loss: 0.1888 | Val Loss: 0.3452 | Val Acc: 83.75% | Patience: 5/20
Epoch  30 | Train Loss: 0.1770 | Val Loss: 0.1224 | Val Acc: 96.42% | Patience: 0/20
Epoch  35 | Train Loss: 0.1784 | Val Loss: 0.6209 | Val Acc: 65.88% | Patience: 5/20
Epoch  40 | Train Loss: 0.1792 | Val Loss: 0.2442 | Val Acc: 90.41% | Patience: 10/20
Epoch  45 | Train Loss: 0.1645 | Val Loss: 0.8853 | Val Acc: 53.07% | Patience: 15/20
Epoch  50 | Train Loss: 0.1834 | Val Loss: 0.3366 | Val Acc: 83.01% | Patience: 20/20

Early stopping at epoch 50

Best model loaded.


In [23]:
with torch.no_grad():

    outputs = model(X_test)

    predictions = outputs.argmax(dim=1)

    accuracy = (predictions == y_test).float().mean()

    print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.9641568064689636


In [27]:
discount_index = df_0.columns.get_loc("Discount")

print(discount_index)

2


In [38]:
import torch
import numpy as np

def get_discount_ranges(
    model,
    sample,
    discount_index,
    min_discount=0.0,
    max_discount=0.8,
    step=0.001
):
    model.eval()

    class_names = {
        0: "Loss",
        1: "Medium Profit",
        2: "High Profit"
    }

    predictions = []

    with torch.no_grad():
        discount = min_discount

        while discount <= max_discount + 1e-9:
            x = sample.clone()
            x[discount_index] = discount

            output = model(x.unsqueeze(0))
            predicted_class = output.argmax(dim=1).item()

            predictions.append(
                (round(discount, 3), predicted_class)
            )

            discount += step

    # Turn predictions into ranges
    ranges = []

    start_discount = predictions[0][0]
    current_class = predictions[0][1]

    for i in range(1, len(predictions)):
        discount, predicted_class = predictions[i]

        # Class changed -> close previous range
        if predicted_class != current_class:
            end_discount = predictions[i - 1][0]

            ranges.append({
                "start": start_discount,
                "end": end_discount,
                "class": current_class
            })

            start_discount = discount
            current_class = predicted_class

    # Add final range
    ranges.append({
        "start": start_discount,
        "end": predictions[-1][0],
        "class": current_class
    })

    return ranges, class_names

In [57]:
sample = X_test[1]

discount_index = df_0.columns.get_loc("Discount")

ranges, class_names = get_discount_ranges(
    model,
    sample,
    discount_index,
    min_discount=0,
    max_discount=0.8,
    step=0.001
)

In [58]:
print("Discount boundaries:\n")

for r in ranges:
    print(
        f"{r['start'] * 100:.1f}% - "
        f"{r['end'] * 100:.1f}%"
        f"  →  {class_names[r['class']]}"
    )

Discount boundaries:

0.0% - 21.5%  →  Medium Profit
21.6% - 80.0%  →  Loss


At discount 0.0% → Medium Profit (confidence 94.9%)
